# 01_EDA.ipynb
## FakeJobShield: Exploratory Data Analysis
This notebook performs Exploratory Data Analysis (EDA) on the fake job postings dataset to understand its characteristics, distribution of labels, missing values, correlation patterns, and other key insights for publication.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import os

# Adjust working directory if run from notebooks folder
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

# Create results dir
os.makedirs("results", exist_ok=True)


In [ ]:
# Load dataset
df = pd.read_csv("data/fake_job_postings.csv")
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())


In [ ]:
# Overview of data types and non-null values
df.info()


In [ ]:
# Target distribution (Class Imbalance)
fraud_counts = df["fraudulent"].value_counts()
print("Target Class Counts:")
print(fraud_counts)

# Normalize fraudulent representations (t/f or 1/0)
df['fraudulent_int'] = df['fraudulent'].map({'t': 1, 'f': 0, '1': 1, '0': 0, 1: 1, 0: 0}).fillna(0).astype(int)
print("\nFraudulent ratio:", df['fraudulent_int'].mean() * 100, "%")

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='fraudulent_int', palette="muted")
plt.title("Distribution of Job Postings (0 = Genuine, 1 = Fraudulent)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("results/class_distribution.png", dpi=150)
plt.show()


In [ ]:
# Missing Value Analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Percentage": missing_pct}).sort_values(by="Missing Count", ascending=False)
print("Missing Values Analysis:")
print(missing_df[missing_df["Missing Count"] > 0])

# Plot missing values heatmap for text columns
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing Values Heatmap")
plt.tight_layout()
plt.savefig("results/missing_values_heatmap.png", dpi=150)
plt.show()


In [ ]:
# Correlation analysis of numeric/binary fields
# Normalize other boolean columns
for col in ["telecommuting", "has_company_logo", "has_questions"]:
    df[col] = df[col].map({'t': 1, 'f': 0, '1': 1, '0': 0, 1: 1, 0: 0, True: 1, False: 0}).fillna(0).astype(int)

numeric_cols = ["telecommuting", "has_company_logo", "has_questions", "fraudulent_int"]
corr = df[numeric_cols].corr()
print("Correlation Matrix:")
print(corr)

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".3f", vmin=-1, vmax=1)
plt.title("Correlation Analysis of Binary Features")
plt.tight_layout()
plt.savefig("results/correlation_matrix.png", dpi=150)
plt.show()


In [ ]:
# Impact of Company Logo on Fraudulence
logo_fraud = pd.crosstab(df["has_company_logo"], df["fraudulent_int"], normalize="index") * 100
print("Logo presence vs Fraudulence (percentage):\n", logo_fraud)

plt.figure(figsize=(6, 4))
sns.barplot(x=logo_fraud.index, y=logo_fraud[1], palette="Blues_d")
plt.title("Fraud Rate vs Presence of Company Logo")
plt.xlabel("Has Company Logo (0=No, 1=Yes)")
plt.ylabel("Fraud Rate (%)")
plt.tight_layout()
plt.savefig("results/logo_impact.png", dpi=150)
plt.show()


In [ ]:
# Impact of Telecommuting on Fraudulence
tele_fraud = pd.crosstab(df["telecommuting"], df["fraudulent_int"], normalize="index") * 100
print("Telecommuting vs Fraudulence (percentage):\n", tele_fraud)

plt.figure(figsize=(6, 4))
sns.barplot(x=tele_fraud.index, y=tele_fraud[1], palette="Oranges_d")
plt.title("Fraud Rate vs Telecommuting Option")
plt.xlabel("Telecommuting (0=No, 1=Yes)")
plt.ylabel("Fraud Rate (%)")
plt.tight_layout()
plt.savefig("results/telecommuting_impact.png", dpi=150)
plt.show()


In [ ]:
# Industry-wise Fraud Analysis (Top 10 industries with highest fraud count)
industry_fraud = df[df["fraudulent_int"] == 1]["industry"].value_counts().head(10)
print("Top 10 Fraudulent Industries (Counts):\n", industry_fraud)

plt.figure(figsize=(10, 5))
sns.barplot(x=industry_fraud.values, y=industry_fraud.index, palette="Reds_r")
plt.title("Top 10 Industries with Highest Fraud Advertisement Counts")
plt.xlabel("Fraud Count")
plt.ylabel("Industry")
plt.tight_layout()
plt.savefig("results/industry_fraud.png", dpi=150)
plt.show()


In [ ]:
# Education-wise Fraud Analysis
edu_fraud = df[df["fraudulent_int"] == 1]["required_education"].value_counts().head(10)
print("Top Fraudulent Required Educations:\n", edu_fraud)

plt.figure(figsize=(10, 5))
sns.barplot(x=edu_fraud.values, y=edu_fraud.index, palette="Purples_r")
plt.title("Top Education Requirements in Fraudulent Job Postings")
plt.xlabel("Fraud Count")
plt.ylabel("Education")
plt.tight_layout()
plt.savefig("results/education_fraud.png", dpi=150)
plt.show()


In [ ]:
# WordClouds for Genuine vs Fraudulent descriptions
genuine_text = " ".join(df[df["fraudulent_int"] == 0]["description"].fillna("").astype(str).head(1000))
fraud_text = " ".join(df[df["fraudulent_int"] == 1]["description"].fillna("").astype(str).head(1000))

# WordCloud for Genuine
wc_gen = WordCloud(width=800, height=400, background_color="white", max_words=100).generate(genuine_text)
plt.figure(figsize=(10, 5))
plt.imshow(wc_gen, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud - Genuine Job Descriptions")
plt.tight_layout()
plt.savefig("results/wordcloud_genuine.png", dpi=150)
plt.show()

# WordCloud for Fraudulent
wc_fraud = WordCloud(width=800, height=400, background_color="black", max_words=100).generate(fraud_text)
plt.figure(figsize=(10, 5))
plt.imshow(wc_fraud, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud - Fraudulent Job Descriptions")
plt.tight_layout()
plt.savefig("results/wordcloud_fraudulent.png", dpi=150)
plt.show()
